# 🤖 Lab 03: BERT 中文情感分析微調

**使用 HuggingFace Transformers 微調 `bert-base-chinese`**

---

### 🎯 學習目標
- 使用 HuggingFace AutoModel 載入預訓練 BERT
- 數據預處理與 tokenize
- 完整微調循環（帶 scheduler + grad clipping）
- 評估和互動測試

### 📊 數據集
**ChnSentiCorp** 子集 (3000 訓練 / 1000 測試)

### 🛠 技術棧
Transformers, PyTorch, Datasets

### ⏱ 預計時間
- GPU: **必須** (Colab T4 免費版足夠)
- 運行時間: ~3 分鐘

### 🚀 GPU 加速
**菜單 → 更改執行階段類型 → 硬件加速器 → GPU**

---

## 1. 安裝

In [ ]:
!pip install -q transformers datasets accelerate
import torch
print(f'✅ PyTorch {torch.__version__}')
print(f'✅ CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   設備: {torch.cuda.get_device_name(0)}')

## 2. 載入數據

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'PingFang TC', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

def load_data():
    try:
        from datasets import load_dataset
        ds_tr = load_dataset('lansinuote/ChnSentiCorp', split='train')
        ds_te = load_dataset('lansinuote/ChnSentiCorp', split='test')
        train_df = pd.DataFrame({'text': ds_tr['text'], 'label': ds_tr['label']}).dropna()
        test_df = pd.DataFrame({'text': ds_te['text'], 'label': ds_te['label']}).dropna()
        train_df = train_df.sample(3000, random_state=42).reset_index(drop=True)
        test_df = test_df.sample(1000, random_state=42).reset_index(drop=True)
    except Exception:
        data_dir = '/content/data' if os.path.exists('/content') else './data'
        if not os.path.exists(f'{data_dir}/sample_train.csv'):
            os.makedirs(data_dir, exist_ok=True)
            !wget -q https://raw.githubusercontent.com/chanxaviersy/nlp-ml-knowledge-base/main/docs/08-labs/03-transformer-sentiment/data/sample_train.csv -O {data_dir}/sample_train.csv
            !wget -q https://raw.githubusercontent.com/chanxaviersy/nlp-ml-knowledge-base/main/docs/08-labs/03-transformer-sentiment/data/sample_test.csv -O {data_dir}/sample_test.csv
        train_df = pd.read_csv(f'{data_dir}/sample_train.csv').dropna()
        test_df = pd.read_csv(f'{data_dir}/sample_test.csv').dropna()
    return train_df, test_df

train_df, test_df = load_data()
print(f'📊 Train: {len(train_df)}, Test: {len(test_df)}')
train_df.head()

## 3. 載入 BERT

In [ ]:
MODEL_NAME = 'bert-base-chinese'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f'🧠 設備: {device}')
print(f'🧠 參數量: {sum(p.numel() for p in model.parameters()):,}')

## 4. 準備 DataLoader

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.enc = tokenizer(list(texts), truncation=True, padding='max_length',
                              max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids': self.enc['input_ids'][idx],
            'attention_mask': self.enc['attention_mask'][idx],
            'label': self.labels[idx],
        }

train_loader = DataLoader(
    SentimentDataset(train_df['text'], train_df['label'], tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE, shuffle=True
)
test_loader = DataLoader(
    SentimentDataset(test_df['text'], test_df['label'], tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE * 2
)
print(f'✅ Train batches: {len(train_loader)}, Test batches: {len(test_loader)}')

## 5. 訓練

In [ ]:
import torch.nn as nn

total_steps = len(train_loader) * EPOCHS
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps,
)

def train_epoch(loader):
    model.train()
    total, correct, loss_sum = 0, 0, 0
    for batch in loader:
        ids = batch['input_ids'].to(device)
        am = batch['attention_mask'].to(device)
        lab = batch['label'].to(device)
        optimizer.zero_grad()
        out = model(input_ids=ids, attention_mask=am, labels=lab)
        out.loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        loss_sum += out.loss.item() * lab.size(0)
        correct += (out.logits.argmax(1) == lab).sum().item()
        total += lab.size(0)
    return loss_sum/total, correct/total

@torch.no_grad()
def evaluate(loader):
    model.eval()
    correct, total = 0, 0
    for batch in loader:
        ids = batch['input_ids'].to(device); am = batch['attention_mask'].to(device); lab = batch['label'].to(device)
        out = model(input_ids=ids, attention_mask=am)
        correct += (out.logits.argmax(1) == lab).sum().item()
        total += lab.size(0)
    return correct/total

history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc = train_epoch(train_loader)
    va_acc = evaluate(test_loader)
    history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc); history['val_acc'].append(va_acc)
    print(f'Epoch {epoch}/{EPOCHS} | loss={tr_loss:.4f} acc={tr_acc:.4f} | val acc={va_acc:.4f}')

## 6. 視覺化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], marker='o', label='Train Loss')
axes[0].set_title('Training Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['train_acc'], marker='o', label='Train Acc')
axes[1].plot(history['val_acc'], marker='s', label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. 互動測試

In [ ]:
import torch.nn.functional as F

def predict(text):
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        out = model(**enc)
        probs = F.softmax(out.logits, dim=-1)[0]
        pred = out.logits.argmax(1).item()
    return ('😊 正面' if pred == 1 else '😞 負面'), probs[pred].item()

for t in [
    '這家餐廳的菜真的很好吃，服務也很棒',
    '完全不符合描述，質量差得離譜',
    '出乎意料的驚喜，強烈推薦購買',
    '退貨流程複雜，客服態度差',
]:
    s, c = predict(t)
    print(f'{t:30s} → {s} ({c:.2%})')

## 🎉 恭喜完成 Lab 03！

**你學到了**：
- ✅ HuggingFace Transformers 微調
- ✅ AdamW + Linear Scheduler with warmup
- ✅ Gradient clipping 防止梯度爆炸
- ✅ 預訓練 BERT 在中文任務的強大能力

### 🚀 下一步

👉 [Lab 04: BERT 命名實體識別](../04-ner-with-bert/lab.ipynb) — 序列標註任務

### 📚 延伸閱讀

- [BERT 論文](https://arxiv.org/abs/1810.04805)
- [Chinese-BERT-wwm](https://github.com/ymcui/Chinese-BERT-wwm)
- [LoRA 微調](../../05-transformers/03-llm-finetuning/README.md)